# COMP5329 — Deep Learning

**Tutorial 12 — Deep Generative Models: From VAEs to One-Step Generation**

**Semester 1, 2026**

### Learning Objectives

By the end of this tutorial you will be able to:

1. Derive the **ELBO** objective for Variational Autoencoders and implement a VAE from scratch, explaining the role of the reparameterisation trick.
2. Implement a **GAN** from scratch and explain the minimax objective, identifying failure modes (mode collapse, training instability).
3. Derive the **DDPM** forward and reverse processes, implementing the training loop and sampling algorithm from scratch.
4. Explain how **DDIM** enables deterministic, accelerated sampling from the same trained diffusion model.
5. Define the **score function** $\nabla_x \log p(x)$ and explain how **denoising score matching** connects to the DDPM training objective.
6. Formulate the diffusion process as a **stochastic differential equation (SDE)** and state the reverse-time SDE, identifying the **Probability Flow ODE** as a deterministic equivalent.
7. Derive the **Flow Matching** objective and implement conditional flow matching from scratch, explaining why straight transport paths are preferable.
8. Explain **Rectified Flow** as iterative path straightening and its connection to optimal transport.
9. Define the **consistency function** and explain how Consistency Models enable single-step generation by enforcing self-consistency along ODE trajectories.
10. Compare all generative paradigms (VAE, GAN, DDPM, DDIM, Score/SDE, Flow Matching, Consistency Models) on training objective, sample quality, sampling speed, and mode coverage.

### Topic Coverage

Week 12 covers **deep generative models: from VAEs to one-step generation**. The full topic list (see `Week12_Self_Study_Deep_Generative_Models.ipynb`) is:

- ✅ **VAE** — ELBO, reparameterisation trick, from-scratch implementation *(tutorial)*
- 📖 **GAN** — minimax objective, mode collapse, training instability *(self-study; WGAN also covered there)*
- 📖 **Wasserstein GAN** — Lipschitz constraint, weight clipping vs gradient penalty *(self-study)*
- ✅ **DDPM** — forward noising, ε-prediction training, reverse sampling *(tutorial, with in-class practice)*
- ✅ **DDIM** — deterministic accelerated sampling, non-Markovian update *(tutorial, with in-class practice)*
- ✅ **Score matching & SDE/PF-ODE** — score functions, stochastic and deterministic formulations *(tutorial, conceptual)*
- ✅ **Flow Matching & Rectified Flow** — straight transport paths, fewer integration steps *(tutorial)*
- ✅ **Consistency Models** — single-step generation via self-consistency *(tutorial, conceptual)*
- 📖 **Grand comparison** — unified view of all paradigms on quality, speed, mode coverage *(self-study)*

Due to time constraints, the tutorial focuses on **DDPM as the pivotal model** — every later method (DDIM, Score/SDE, Flow Matching, Consistency) is presented as a refinement or reinterpretation of DDPM's core idea. GAN/WGAN are left as self-study since they have a different conceptual lineage; the self-study notebook works through them fully.

The live session is organised into three parts: **Part A** — tutor walkthrough, **Part B** — in-class coding practice, **Part C** — exam-style Q&A.

### The conceptual thread the tutor will follow

The notebook is intentionally long — the tutor will *not* re-derive every model. Instead, the walkthrough covers:

1. **Why generation is hard.** $\log p_\theta(x)=\log\int p_\theta(x|z)p(z)\,dz$ is intractable. Every model in this notebook is a different relaxation of this integral.
2. **VAE essentials.** ELBO = reconstruction − KL. The **reparameterisation trick** $z=\mu+\sigma\odot\varepsilon$ is the *only* reason gradients reach the encoder — emphasise *why* the gradient cannot flow through a raw `torch.normal(mu, sigma)` call.
3. **DDPM essentials.** Closed-form forward $x_t=\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\varepsilon$; train by predicting $\varepsilon$; sampling reverses the chain.
4. **The unification.** $\varepsilon$-prediction is score matching: $\nabla_{x_t}\log q(x_t|x_0)=-\varepsilon/\sqrt{1-\bar\alpha_t}$. SDE ⇒ PF-ODE ⇒ generation = solving an ODE from noise to data.
5. **Efficiency frontier.** Flow Matching learns straighter paths ⇒ fewer steps. Consistency Models enforce $f(x_t,t)=x_0$ for *all* $t$ on the same trajectory ⇒ one-step sampling. The whole story is *recovering single-step speed* without losing diffusion-quality.
6. **Bridge to practice.** Part B exercises items (3) and the DDIM half of (5).

> Phases 3, 4, 5 in the notebook are read-along material — the tutor names the key result for each and moves on. Students who want depth can run those cells at home.

---
# Part A · Tutor Walkthrough

## 0. The Story So Far and Where We Are Going

In Weeks 7-9 we built models that *understand* and *process* data -- Transformers for sequences, VLMs for multi-modal understanding. But all of these are **discriminative** or **predictive**: they map inputs to outputs. This week we ask a fundamentally different question:

> **How do we generate entirely new, realistic data from noise?**

We will explore five progressively deeper answers, each building on the limitations of the previous:

| Phase | Models | Core Question | Code Depth |
|---|---|---|---|
| 1 | VAE, GAN | Classical approaches to generation? | From-scratch (both) |
| 2 | DDPM, DDIM | Can iterative denoising do better? | From-scratch (DDPM + DDIM sampling) |
| 3 | Score/SDE/ODE | What is the mathematical essence? | Conceptual + visualisation |
| 4 | Flow Matching, Rectified Flow | Can we find straighter paths? | From-scratch (Flow Matching) |
| 5 | Consistency Models | Can we skip iteration entirely? | Conceptual + light code |

**Efficiency trajectory**: VAE/GAN (1 step, limited) $\to$ DDPM (1000 steps) $\to$ DDIM (50 steps) $\to$ Flow Matching (10-50 steps) $\to$ Consistency (1-2 steps).

All five phases use the **same 2D dataset** (8 Gaussians) for direct visual comparison.

In [ ]:
# ── Imports and shared 2D dataset ───────────────────────────────────────────
import math
import time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

%matplotlib inline

# ── Shared 2D dataset: 8 Gaussians in a circle ───────────────────────────
def make_8gaussians(n_samples=10000, std=0.05):
    """Generate 2D data from 8 Gaussians arranged in a circle."""
    angles = torch.linspace(0, 2 * math.pi, 9)[:-1]  # 8 modes
    centres = torch.stack([torch.cos(angles), torch.sin(angles)], dim=1) * 2.0
    # Assign each sample to a random mode
    idx = torch.randint(0, 8, (n_samples,))
    data = centres[idx] + torch.randn(n_samples, 2) * std
    return data

torch.manual_seed(42)
data_2d = make_8gaussians(10000)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(data_2d[:, 0].numpy(), data_2d[:, 1].numpy(), s=1, alpha=0.3)
ax.set_title('Target Distribution: 8 Gaussians')
ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5)
ax.set_aspect('equal')
plt.tight_layout(); plt.show()

---

## Phase 1: Foundations -- VAE and GAN

### 1.1 VAE: Variational Autoencoder

**The problem**: We want to learn a generative model $p_\theta(x)$ that we can sample from. The natural approach is to introduce a latent variable $z$ and maximise the **marginal likelihood**:

$$\log p_\theta(x) = \log \int p_\theta(x|z)\, p(z)\, dz$$

But this integral is **intractable** -- it sums over all possible latent codes.

**The ELBO** (Evidence Lower Bound): Introduce an approximate posterior $q_\phi(z|x)$ and derive a tractable lower bound:

$$\log p_\theta(x) \geq \underbrace{\mathbb{E}_{q_\phi(z|x)}[\log p_\theta(x|z)]}_{\text{Reconstruction}} - \underbrace{D_{\text{KL}}(q_\phi(z|x) \| p(z))}_{\text{KL regularisation}} = \text{ELBO}$$

- **Reconstruction term**: The decoder should reconstruct $x$ from $z$ sampled from the encoder.
- **KL term**: The encoder's posterior should stay close to the prior $p(z) = \mathcal{N}(0, I)$.

**Reparameterisation trick** (enables backprop through sampling):

$$z = \mu_\phi(x) + \sigma_\phi(x) \odot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

**Closed-form KL** for two Gaussians:

$$D_{\text{KL}}(\mathcal{N}(\mu, \sigma^2) \| \mathcal{N}(0, 1)) = \frac{1}{2}\sum_{j=1}^{d}(\mu_j^2 + \sigma_j^2 - \log \sigma_j^2 - 1)$$

In [ ]:
# ── VAE implementation ───────────────────────────────────────────────────────

class VAE(nn.Module):
    def __init__(self, data_dim=2, latent_dim=2, hidden_dim=128):
        super().__init__()
        # Encoder: data -> (mu, logvar)
        self.enc = nn.Sequential(nn.Linear(data_dim, hidden_dim), nn.ReLU(),
                                  nn.Linear(hidden_dim, hidden_dim), nn.ReLU())
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        # Decoder: z -> data
        self.dec = nn.Sequential(nn.Linear(latent_dim, hidden_dim), nn.ReLU(),
                                  nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
                                  nn.Linear(hidden_dim, data_dim))

    def encode(self, x):
        h = self.enc(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterise(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + std * eps

    def decode(self, z):
        return self.dec(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterise(mu, logvar)
        return self.decode(z), mu, logvar


def vae_loss(x_recon, x, mu, logvar):
    """Negative ELBO = reconstruction + KL."""
    recon = F.mse_loss(x_recon, x, reduction='sum') / x.size(0)
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
    return recon + kl

In [ ]:
# ── VAE training ───────────────────────────────────────────────────────────
torch.manual_seed(42)
vae = VAE()
opt_vae = torch.optim.Adam(vae.parameters(), lr=1e-3)

for epoch in range(300):
    idx = torch.randperm(len(data_2d))[:256]
    x_batch = data_2d[idx]
    x_recon, mu, logvar = vae(x_batch)
    loss = vae_loss(x_recon, x_batch, mu, logvar)
    opt_vae.zero_grad(); loss.backward(); opt_vae.step()
    if (epoch + 1) % 100 == 0:
        print(f'  VAE epoch {epoch+1}, loss: {loss.item():.2f}')

In [ ]:
# ── VAE visualisation ────────────────────────────────────────────────────────
with torch.no_grad():
    z_samples = torch.randn(2000, 2)
    vae_samples = vae.decode(z_samples).numpy()
    mu_all, _ = vae.encode(data_2d[:2000])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].scatter(data_2d[:2000, 0], data_2d[:2000, 1], s=1, alpha=0.3)
axes[0].set_title('Real Data'); axes[0].set_xlim(-3.5, 3.5); axes[0].set_ylim(-3.5, 3.5)
axes[1].scatter(vae_samples[:, 0], vae_samples[:, 1], s=1, alpha=0.3, c='orange')
axes[1].set_title('VAE Samples (blurry!)'); axes[1].set_xlim(-3.5, 3.5); axes[1].set_ylim(-3.5, 3.5)
axes[2].scatter(mu_all[:, 0].numpy(), mu_all[:, 1].numpy(), s=1, alpha=0.3, c='green')
axes[2].set_title('Latent Space')
for ax in axes: ax.set_aspect('equal')
plt.tight_layout(); plt.show()
print('Note: VAE samples fill the gaps between modes -- characteristic blurriness.')

> **Transition**: The VAE produces a structured latent space and stable training, but its samples are blurry -- the reconstruction loss averages over modes. Can we get **sharper** samples by abandoning explicit density estimation?

### 1.2 GAN: Generative Adversarial Network

**Minimax objective** -- a two-player game:

$$\min_G \max_D \; \mathbb{E}_{x \sim p_{\text{data}}}[\log D(x)] + \mathbb{E}_{z \sim p(z)}[\log(1 - D(G(z)))]$$

- **Generator** $G$: maps noise $z$ to fake data. Tries to fool the discriminator.
- **Discriminator** $D$: classifies real vs. fake. Tries to catch the generator.

**Known failure modes**: Mode collapse (generator ignores some modes), training instability (D and G oscillate), vanishing gradients (D too strong $\Rightarrow$ G gets no signal).

### 1.3 VAE vs. GAN Comparison

| Aspect | VAE | GAN |
|---|---|---|
| Training objective | Maximise ELBO | Minimax game |
| Density estimation | Explicit (approximate) | Implicit |
| Sample quality | Blurry | Sharp |
| Training stability | Stable | Unstable |
| Mode coverage | Good (KL prevents collapse) | Mode collapse risk |
| Latent space | Structured, interpolable | Unstructured |

> **Transition**: VAEs give us principled training but blurry samples. GANs give us sharp samples but unstable training and mode collapse. Can we have **both** -- principled, stable training that produces sharp, high-quality samples with full mode coverage? The answer came in 2020 with **diffusion models**, which approached the problem from an entirely different angle: what if generation is just *iterative denoising*?

---

## Phase 2: Diffusion Models -- DDPM and DDIM

### 2.1 DDPM: Denoising Diffusion Probabilistic Models

**Core insight**: Instead of learning to generate in one shot, learn to **gradually remove noise**. Start with pure noise, iteratively denoise to get a clean sample.

**Forward process** (fixed, adds noise gradually):

$$q(x_t | x_0) = \mathcal{N}\big(x_t;\; \sqrt{\bar{\alpha}_t}\, x_0,\; (1 - \bar{\alpha}_t)\, I\big)$$

which means we can sample $x_t$ directly: $\;x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\, \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$

where $\alpha_t = 1 - \beta_t$ and $\bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s$.

**Reverse process** (learned, removes noise):

$$p_\theta(x_{t-1}|x_t) = \mathcal{N}\big(x_{t-1};\; \mu_\theta(x_t, t),\; \sigma_t^2 I\big)$$

**Noise prediction reparameterisation**: Instead of predicting $\mu_\theta$, predict the noise $\epsilon_\theta(x_t, t)$:

$$\mu_\theta(x_t, t) = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}}\, \epsilon_\theta(x_t, t)\right)$$

**Simplified training objective**:

$$\mathcal{L}_{\text{simple}} = \mathbb{E}_{t,\, x_0,\, \epsilon}\!\left[\|\epsilon - \epsilon_\theta(x_t, t)\|^2\right]$$

Remarkably simple: sample a timestep, add noise, predict the noise.

In [ ]:
# ── DDPM noise schedule ──────────────────────────────────────────────────────

T_diffusion = 300  # fewer steps for 2D (not 1000 -- sufficient for toy data)

def linear_beta_schedule(T, beta_start=1e-4, beta_end=0.02):
    return torch.linspace(beta_start, beta_end, T)

betas = linear_beta_schedule(T_diffusion)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

# Plot alpha_bar decay
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(alpha_bars.numpy())
ax.set_xlabel('Timestep t'); ax.set_ylabel(r'$\bar{\alpha}_t$')
ax.set_title(r'Noise schedule: $\bar{\alpha}_t$ decays from 1 (clean) to ~0 (noise)')
ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# ── Forward process visualisation ───────────────────────────────────────────
x0 = data_2d[:500]
timesteps = [0, 30, 75, 150, 225, T_diffusion - 1]

fig, axes = plt.subplots(1, len(timesteps), figsize=(18, 3))
for i, t in enumerate(timesteps):
    ab = alpha_bars[t]
    xt = math.sqrt(ab) * x0 + math.sqrt(1 - ab) * torch.randn_like(x0)
    axes[i].scatter(xt[:, 0].numpy(), xt[:, 1].numpy(), s=1, alpha=0.3)
    axes[i].set_title(f't = {t}\n' + r'$\bar{\alpha}$' + f' = {ab:.3f}')
    axes[i].set_xlim(-4, 4); axes[i].set_ylim(-4, 4); axes[i].set_aspect('equal')
plt.suptitle('Forward Process: Data dissolves into noise', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# ── DDPM noise predictor + training ───────────────────────────────────────

class SinusoidalTimeEmb(nn.Module):
    """Sinusoidal timestep embedding (same idea as positional encoding in Week 7)."""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        args = t.unsqueeze(-1) * freqs.unsqueeze(0)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

class NoisePredictor(nn.Module):
    """MLP that predicts noise given (x_t, t)."""
    def __init__(self, data_dim=2, hidden_dim=128, time_dim=32):
        super().__init__()
        self.time_emb = SinusoidalTimeEmb(time_dim)
        self.net = nn.Sequential(
            nn.Linear(data_dim + time_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, data_dim))
    def forward(self, x, t):
        t_emb = self.time_emb(t)     # (B, time_dim)
        return self.net(torch.cat([x, t_emb], dim=-1))  # (B, data_dim)

# Training
torch.manual_seed(42)
ddpm_model = NoisePredictor()
opt_ddpm = torch.optim.Adam(ddpm_model.parameters(), lr=1e-3)

for step in range(5000):
    idx = torch.randperm(len(data_2d))[:256]
    x0 = data_2d[idx]                                        # (B, 2)
    t = torch.randint(0, T_diffusion, (256,))                # (B,)
    eps = torch.randn_like(x0)                                # (B, 2)
    ab = alpha_bars[t].unsqueeze(-1)                          # (B, 1)
    xt = torch.sqrt(ab) * x0 + torch.sqrt(1 - ab) * eps      # (B, 2)
    eps_pred = ddpm_model(xt, t.float())                      # (B, 2)
    loss = F.mse_loss(eps_pred, eps)
    opt_ddpm.zero_grad(); loss.backward(); opt_ddpm.step()
    if (step + 1) % 1000 == 0:
        print(f'  DDPM step {step+1}, loss: {loss.item():.4f}')

### 2.2 DDPM Sampling

```
Algorithm: DDPM Sampling
x_T ~ N(0, I)
for t = T, T-1, ..., 1:
    z ~ N(0, I) if t > 1, else z = 0
    x_{t-1} = (1/sqrt(alpha_t)) * (x_t - beta_t/sqrt(1 - alpha_bar_t) * eps_theta(x_t, t)) + sigma_t * z
return x_0
```

In [ ]:
# ── DDPM sampling ───────────────────────────────────────────────────────────

@torch.no_grad()
def ddpm_sample(model, n_samples, T, betas, alpha_bars):
    alphas = 1.0 - betas
    x = torch.randn(n_samples, 2)  # start from noise
    trajectory = [x.clone()]
    for t in reversed(range(T)):
        t_batch = torch.full((n_samples,), t, dtype=torch.float)
        eps_pred = model(x, t_batch)
        ab = alpha_bars[t]
        a = alphas[t]
        b = betas[t]
        mu = (1 / math.sqrt(a)) * (x - (b / math.sqrt(1 - ab)) * eps_pred)
        if t > 0:
            sigma = math.sqrt(b)
            x = mu + sigma * torch.randn_like(x)
        else:
            x = mu
        if t % (T // 8) == 0:
            trajectory.append(x.clone())
    return x, trajectory

t0 = time.time()
ddpm_samples, ddpm_traj = ddpm_sample(ddpm_model, 2000, T_diffusion, betas, alpha_bars)
ddpm_time = time.time() - t0
print(f'DDPM sampling ({T_diffusion} steps): {ddpm_time:.2f}s')

In [ ]:
# ── DDPM results ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].scatter(data_2d[:2000, 0], data_2d[:2000, 1], s=1, alpha=0.3, label='Real')
axes[0].scatter(ddpm_samples[:, 0].numpy(), ddpm_samples[:, 1].numpy(), s=1, alpha=0.3, c='purple', label='DDPM')
axes[0].legend(); axes[0].set_title(f'DDPM Samples ({T_diffusion} steps)')
axes[0].set_xlim(-3.5, 3.5); axes[0].set_ylim(-3.5, 3.5); axes[0].set_aspect('equal')

# Denoising trajectory for a few samples
for j in range(5):
    traj_x = [ddpm_traj[i][j, 0].item() for i in range(len(ddpm_traj))]
    traj_y = [ddpm_traj[i][j, 1].item() for i in range(len(ddpm_traj))]
    axes[1].plot(traj_x, traj_y, '-o', markersize=2, alpha=0.6)
axes[1].set_title('Denoising Trajectories (noise → data)')
axes[1].set_xlim(-4, 4); axes[1].set_ylim(-4, 4); axes[1].set_aspect('equal')
plt.tight_layout(); plt.show()
print('DDPM produces high-quality samples -- all 8 modes covered! But requires many steps.')

### 2.3 DDIM: Deterministic Accelerated Sampling

> **Transition**: DDPM produces excellent samples, but requires $T$ sequential denoising steps. Can we sample faster from the **same trained model**?

**DDIM update rule** (Song et al., 2020):

$$x_{t-1} = \sqrt{\bar{\alpha}_{t-1}} \underbrace{\left(\frac{x_t - \sqrt{1-\bar{\alpha}_t}\, \epsilon_\theta}{\sqrt{\bar{\alpha}_t}}\right)}_{\text{predicted } x_0} + \sqrt{1-\bar{\alpha}_{t-1} - \sigma_t^2} \cdot \epsilon_\theta + \sigma_t \epsilon$$

When $\sigma_t = 0$: **deterministic** sampling, and we can **skip timesteps** (use a subsequence with $S \ll T$ steps). Same trained model, no retraining!

In [ ]:
# ── DDIM sampling ───────────────────────────────────────────────────────────

@torch.no_grad()
def ddim_sample(model, n_samples, T, alpha_bars, num_steps=50, eta=0.0):
    """DDIM sampling with S << T steps. eta=0 is deterministic."""
    # Create sub-sequence of timesteps
    step_size = T // num_steps
    timesteps = list(range(T - 1, -1, -step_size))[:num_steps]
    x = torch.randn(n_samples, 2)
    for i, t in enumerate(timesteps):
        t_batch = torch.full((n_samples,), t, dtype=torch.float)
        eps_pred = model(x, t_batch)
        ab_t = alpha_bars[t]
        ab_prev = alpha_bars[timesteps[i + 1]] if i + 1 < len(timesteps) else torch.tensor(1.0)
        # Predicted x_0
        x0_pred = (x - math.sqrt(1 - ab_t) * eps_pred) / math.sqrt(ab_t)
        # DDIM update
        sigma = eta * math.sqrt((1 - ab_prev) / (1 - ab_t)) * math.sqrt(1 - ab_t / ab_prev)
        dir_xt = math.sqrt(1 - ab_prev - sigma**2) * eps_pred
        x = math.sqrt(ab_prev) * x0_pred + dir_xt
        if sigma > 0:
            x = x + sigma * torch.randn_like(x)
    return x

# Compare different step counts
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
configs = [('DDPM', T_diffusion), ('DDIM-50', 50), ('DDIM-20', 20), ('DDIM-10', 10)]
times_compare = []
for ax, (name, steps) in zip(axes, configs):
    t0 = time.time()
    if name == 'DDPM':
        samples, _ = ddpm_sample(ddpm_model, 2000, T_diffusion, betas, alpha_bars)
    else:
        samples = ddim_sample(ddpm_model, 2000, T_diffusion, alpha_bars, num_steps=steps)
    elapsed = time.time() - t0
    times_compare.append((name, elapsed))
    ax.scatter(samples[:, 0].numpy(), samples[:, 1].numpy(), s=1, alpha=0.3)
    ax.set_title(f'{name}\n{elapsed:.2f}s'); ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5); ax.set_aspect('equal')
plt.suptitle('Same model, different sampling speeds', fontsize=13)
plt.tight_layout(); plt.show()
for name, t in times_compare: print(f'  {name}: {t:.3f}s')

> **Transition**: DDPM/DDIM work remarkably well, but we treated them as **discrete processes** with a fixed number of steps. What happens if we take the step size to zero and think in **continuous time**? This leads to a profound unification through stochastic differential equations.

---

## Phase 3: The Continuous View -- Score Functions and SDEs

### 3.1 The Score Function

**Definition**: The **score** of a distribution $p(x)$ is the gradient of its log-density:

$$s(x) = \nabla_x \log p(x)$$

It points in the direction of **increasing density** -- toward the modes of the distribution.

**Example**: For a Gaussian $\mathcal{N}(\mu, \sigma^2 I)$: $\;s(x) = -(x - \mu) / \sigma^2$ -- an arrow pointing toward the mean.

**Connection to DDPM**: The noise predictor $\epsilon_\theta(x_t, t)$ is directly related to the score:

$$s_\theta(x_t, t) = -\frac{\epsilon_\theta(x_t, t)}{\sqrt{1 - \bar{\alpha}_t}}$$

**The DDPM training objective IS denoising score matching!** Predicting noise = estimating the score.

In [ ]:
# ── Score field visualisation ────────────────────────────────────────────────
# Compute the score of the 8-Gaussian mixture analytically
def score_8gaussians(x, std=0.05):
    """Analytic score of 8-Gaussian mixture."""
    angles = torch.linspace(0, 2 * math.pi, 9)[:-1]
    centres = torch.stack([torch.cos(angles), torch.sin(angles)], dim=1) * 2.0  # (8, 2)
    # Compute unnormalised density contributions
    diffs = x.unsqueeze(1) - centres.unsqueeze(0)  # (N, 8, 2)
    log_ps = -0.5 * (diffs ** 2).sum(-1) / std**2  # (N, 8)
    weights = F.softmax(log_ps, dim=1)               # (N, 8)
    # Weighted sum of per-component scores
    scores = -(weights.unsqueeze(-1) * diffs).sum(1) / std**2  # (N, 2)
    return scores

# Grid
grid_x = torch.linspace(-3.5, 3.5, 20)
grid_y = torch.linspace(-3.5, 3.5, 20)
xx, yy = torch.meshgrid(grid_x, grid_y, indexing='xy')
grid_pts = torch.stack([xx.flatten(), yy.flatten()], dim=1)
scores = score_8gaussians(grid_pts)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(data_2d[:2000, 0], data_2d[:2000, 1], s=1, alpha=0.1, c='gray')
ax.quiver(grid_pts[:, 0].numpy(), grid_pts[:, 1].numpy(),
          scores[:, 0].numpy(), scores[:, 1].numpy(),
          color='blue', alpha=0.7, scale=200)
ax.set_title('Score Field: arrows point toward data modes', fontsize=12)
ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5); ax.set_aspect('equal')
plt.tight_layout(); plt.show()

### 3.2 SDE Formulation

In continuous time, the forward process becomes an **SDE** (Stochastic Differential Equation):

$$dx = f(x, t)\, dt + g(t)\, dw$$

The remarkable result (**Anderson, 1982**): the reverse-time process is also an SDE:

$$dx = \left[f(x,t) - g(t)^2 \nabla_x \log p_t(x)\right] dt + g(t)\, d\bar{w}$$

And there exists a deterministic **Probability Flow ODE** with the **same marginal distributions**:

$$dx = \left[f(x,t) - \frac{1}{2}g(t)^2 \nabla_x \log p_t(x)\right] dt$$

### 3.3 The Unification

```
DDPM (discrete, stochastic)    ↔  Reverse-time SDE (continuous, stochastic)
       ↕                                      ↕
DDIM (discrete, deterministic) ↔  Probability Flow ODE (continuous, deterministic)
```

DDPM is a discretisation of the reverse SDE. DDIM is a discretisation of the PF-ODE.

> **Key insight**: Generation = solving an ODE where the velocity field is determined by the learned score function. This ODE view opens the door to Phase 4.

In [ ]:
# ── ODE vs SDE trajectory comparison ─────────────────────────────────────
# Simulate SDE (stochastic) vs ODE (deterministic) paths using trained DDPM
torch.manual_seed(123)
n_show = 8
x_start = torch.randn(n_show, 2) * 2.5

def simulate_reverse(model, x_init, T, alpha_bars, betas, stochastic=True, n_steps=200):
    step_size = max(1, T // n_steps)
    timesteps = list(range(T - 1, -1, -step_size))[:n_steps]
    x = x_init.clone()
    path = [x.clone()]
    for t in timesteps:
        t_b = torch.full((x.size(0),), t, dtype=torch.float)
        eps = model(x, t_b)
        ab = alpha_bars[t]
        a = 1.0 - betas[t]
        mu = (1/math.sqrt(a)) * (x - (betas[t]/math.sqrt(1-ab)) * eps)
        if stochastic and t > 0:
            x = mu + math.sqrt(betas[t]) * torch.randn_like(x)
        else:
            x = mu
        path.append(x.clone())
    return path

with torch.no_grad():
    sde_path = simulate_reverse(ddpm_model, x_start, T_diffusion, alpha_bars, betas, stochastic=True)
    ode_path = simulate_reverse(ddpm_model, x_start, T_diffusion, alpha_bars, betas, stochastic=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
for j in range(n_show):
    sx = [sde_path[i][j,0].item() for i in range(len(sde_path))]
    sy = [sde_path[i][j,1].item() for i in range(len(sde_path))]
    ax1.plot(sx, sy, alpha=0.5)
ax1.scatter(data_2d[:1000,0], data_2d[:1000,1], s=1, alpha=0.05, c='gray')
ax1.set_title('SDE Trajectories (stochastic, noisy paths)'); ax1.set_xlim(-4,4); ax1.set_ylim(-4,4); ax1.set_aspect('equal')

for j in range(n_show):
    ox = [ode_path[i][j,0].item() for i in range(len(ode_path))]
    oy = [ode_path[i][j,1].item() for i in range(len(ode_path))]
    ax2.plot(ox, oy, alpha=0.5)
ax2.scatter(data_2d[:1000,0], data_2d[:1000,1], s=1, alpha=0.05, c='gray')
ax2.set_title('ODE Trajectories (deterministic, smooth paths)'); ax2.set_xlim(-4,4); ax2.set_ylim(-4,4); ax2.set_aspect('equal')
plt.tight_layout(); plt.show()
print('Both converge to data modes. ODE paths are smooth but CURVED → many steps needed.')

> **Transition**: We now understand that generation = solving an ODE from noise to data. The score function defines the velocity field. But these paths are often **curved**, requiring many steps. What if we directly learned a velocity field that transports noise to data along **straight lines**?

---

## Phase 4: The Transport View -- Flow Matching and Rectified Flow

### 4.1 Flow Matching: Straight Paths from Noise to Data

**Paradigm shift**: Instead of (1) define a noising process, (2) learn to reverse it -- we directly ask: what is the simplest way to move a noise distribution to a data distribution?

Define a velocity field $v_\theta(x, t)$ that generates a flow from $t=0$ (noise) to $t=1$ (data):

$$\frac{dx}{dt} = v_\theta(x, t)$$

**Conditional Flow Matching** (CFM) -- the practical training objective:

For each training pair $(x_0 \sim \mathcal{N}(0, I),\; x_1 \sim p_{\text{data}})$, define a **straight-line** interpolation:

$$x_t = (1-t)\, x_0 + t\, x_1$$

The conditional velocity is simply: $u_t(x \mid x_1) = x_1 - x_0$ (direction from noise to data).

**Training objective**:

$$\mathcal{L}_{\text{CFM}} = \mathbb{E}_{t,\, x_0,\, x_1}\!\left[\|v_\theta(x_t, t) - (x_1 - x_0)\|^2\right]$$

> **Simplicity**: DDPM predicts the noise $\epsilon$ added to $x_0$. CFM predicts the direction $x_1 - x_0$ from noise to data. Both are MSE losses. But CFM's paths are **straight by construction**.

In [ ]:
# ── Flow Matching: velocity field + training ─────────────────────────────

class VelocityField(nn.Module):
    """Predict velocity v(x_t, t) for flow matching."""
    def __init__(self, data_dim=2, hidden_dim=128, time_dim=32):
        super().__init__()
        self.time_emb = SinusoidalTimeEmb(time_dim)  # reuse from Phase 2
        self.net = nn.Sequential(
            nn.Linear(data_dim + time_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, data_dim))
    def forward(self, x, t):
        t_emb = self.time_emb(t * 100)  # scale t to match timestep range
        return self.net(torch.cat([x, t_emb], dim=-1))

# Training
torch.manual_seed(42)
fm_model = VelocityField()
opt_fm = torch.optim.Adam(fm_model.parameters(), lr=1e-3)

for step in range(5000):
    idx = torch.randperm(len(data_2d))[:256]
    x1 = data_2d[idx]                          # data points
    x0 = torch.randn_like(x1)                  # noise
    t = torch.rand(256)                         # t ~ U(0, 1)
    xt = (1 - t.unsqueeze(-1)) * x0 + t.unsqueeze(-1) * x1  # straight-line interpolation
    target = x1 - x0                            # target velocity
    v_pred = fm_model(xt, t)
    loss = F.mse_loss(v_pred, target)
    opt_fm.zero_grad(); loss.backward(); opt_fm.step()
    if (step + 1) % 1000 == 0:
        print(f'  Flow Matching step {step+1}, loss: {loss.item():.4f}')

In [ ]:
# ── Flow Matching: Euler ODE sampling ─────────────────────────────────────

@torch.no_grad()
def fm_sample(model, n_samples, num_steps=100):
    """Sample by solving dx/dt = v(x,t) from t=0 to t=1 via Euler."""
    x = torch.randn(n_samples, 2)
    dt = 1.0 / num_steps
    trajectory = [x.clone()]
    for i in range(num_steps):
        t = torch.full((n_samples,), i * dt)
        x = x + model(x, t) * dt
        if i % (num_steps // 8) == 0:
            trajectory.append(x.clone())
    trajectory.append(x.clone())
    return x, trajectory

t0 = time.time()
fm_samples, fm_traj = fm_sample(fm_model, 2000, num_steps=50)
fm_time = time.time() - t0
print(f'Flow Matching sampling (50 steps): {fm_time:.2f}s')

In [ ]:
# ── Flow Matching visualisation ─────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Flow trajectories
for j in range(12):
    fx = [fm_traj[i][j, 0].item() for i in range(len(fm_traj))]
    fy = [fm_traj[i][j, 1].item() for i in range(len(fm_traj))]
    ax1.plot(fx, fy, '-o', markersize=2, alpha=0.5)
ax1.scatter(data_2d[:1000, 0], data_2d[:1000, 1], s=1, alpha=0.05, c='gray')
ax1.set_title('Flow Matching Trajectories (approximately STRAIGHT!)')
ax1.set_xlim(-4, 4); ax1.set_ylim(-4, 4); ax1.set_aspect('equal')

# Samples
ax2.scatter(data_2d[:2000, 0], data_2d[:2000, 1], s=1, alpha=0.2, label='Real')
ax2.scatter(fm_samples[:, 0].numpy(), fm_samples[:, 1].numpy(), s=1, alpha=0.2, c='green', label='FM')
ax2.legend(); ax2.set_title('Flow Matching Samples (50 steps)')
ax2.set_xlim(-3.5, 3.5); ax2.set_ylim(-3.5, 3.5); ax2.set_aspect('equal')
plt.tight_layout(); plt.show()
print('Paths are much straighter than diffusion ODE paths → fewer steps needed!')

### 4.2 Rectified Flow: Straightening Paths Further

Even with CFM, paths may not be perfectly straight because different $(x_0, x_1)$ pairs create **crossing paths**.

**Rectified Flow** (Liu et al., 2023) iteratively straightens them:
1. Train a flow model $v_1$. Use it to generate coupled pairs $(x_0^{(i)}, x_1^{(i)})$ by solving the ODE.
2. Retrain on these new pairs. The correspondence is now straighter.
3. Repeat. Each iteration straightens further.

**Why straightness matters**: Straighter paths can be traversed with fewer Euler steps. In the limit of perfectly straight paths, **one step suffices**.

**Connection to optimal transport**: Perfectly straight, non-crossing paths correspond to the OT map between noise and data distributions.

In [ ]:
# ── Step count vs quality comparison ───────────────────────────────────────
step_counts = [1, 5, 10, 20, 50, 100]
fig, axes = plt.subplots(1, len(step_counts), figsize=(18, 3))
for ax, s in zip(axes, step_counts):
    samples, _ = fm_sample(fm_model, 1000, num_steps=s)
    ax.scatter(samples[:, 0].numpy(), samples[:, 1].numpy(), s=1, alpha=0.3)
    ax.set_title(f'{s} step{"s" if s > 1 else ""}')
    ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5); ax.set_aspect('equal')
plt.suptitle('Flow Matching: Quality vs Number of Euler Steps', fontsize=13)
plt.tight_layout(); plt.show()
print('Flow Matching degrades gracefully -- even 10 steps gives recognisable modes.')

| Method | Path shape | Steps needed | Training |
|---|---|---|---|
| DDPM / SDE | Curved + stochastic | ~1000 | Denoise |
| DDIM / PF-ODE | Curved + deterministic | ~50-100 | Same model as DDPM |
| Flow Matching | Approximately straight | ~10-50 | Velocity regression |
| Rectified Flow | Straighter | ~5-10 | Iterative reflow |

> **Transition**: Rectified Flow showed that straighter paths need fewer steps. In the limit, perfectly straight paths need one step. But can we *guarantee* single-step generation without requiring perfectly straight paths? **Consistency Models** answer yes, by enforcing a different property: all points on the same trajectory must map to the same output.

---

## Phase 5: Efficiency Frontier -- Consistency Models

### 5.1 The Self-Consistency Property

**Consistency function**: $f: (x_t, t) \mapsto x_0$ -- maps any point on an ODE trajectory to the trajectory's **origin** (the clean data point).

**Defining constraint** (self-consistency):

$$f(x_t, t) = f(x_{t'}, t') \quad \forall\, t, t' \in [0, T]$$

for any $(x_t, x_{t'})$ on the **same** PF-ODE trajectory.

**Boundary condition**: $f(x_0, 0) = x_0$ (identity at $t=0$). Enforced architecturally:

$$f_\theta(x, t) = c_{\text{skip}}(t) \cdot x + c_{\text{out}}(t) \cdot F_\theta(x, t)$$

where $c_{\text{skip}}(0) = 1$, $c_{\text{out}}(0) = 0$.

**Consistency Distillation** (given a pre-trained diffusion model):

$$\mathcal{L}_{\text{CD}} = \mathbb{E}\!\left[d\!\left(f_\theta(x_{t_{n+1}}, t_{n+1}),\; f_{\theta^-}(\hat{x}_{t_n}, t_n)\right)\right]$$

where $\hat{x}_{t_n}$ is one ODE step from $x_{t_{n+1}}$ using the pre-trained model, and $\theta^-$ is an EMA of $\theta$.

**Single-step generation**: Just evaluate $f_\theta(x_T, T)$ once! No iteration needed.

In [ ]:
# ── Consistency model concept visualisation ───────────────────────────────
# Show ODE trajectories and illustrate the consistency property
torch.manual_seed(42)
n_traj = 6
x_init = torch.randn(n_traj, 2) * 2.5

with torch.no_grad():
    paths = simulate_reverse(ddpm_model, x_init, T_diffusion, alpha_bars, betas, stochastic=False, n_steps=100)

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(data_2d[:1000, 0], data_2d[:1000, 1], s=1, alpha=0.05, c='gray')
colors = plt.cm.Set1(np.linspace(0, 1, n_traj))
for j in range(n_traj):
    px = [paths[i][j, 0].item() for i in range(len(paths))]
    py = [paths[i][j, 1].item() for i in range(len(paths))]
    ax.plot(px, py, '-', color=colors[j], alpha=0.6, linewidth=2)
    # Mark several points on the trajectory
    for k in [0, len(paths)//4, len(paths)//2, 3*len(paths)//4, -1]:
        ax.plot(paths[k][j, 0].item(), paths[k][j, 1].item(), 'o',
                color=colors[j], markersize=5)
    # Endpoint (x_0)
    ax.plot(paths[-1][j, 0].item(), paths[-1][j, 1].item(), '*',
            color=colors[j], markersize=15, markeredgecolor='black')

ax.set_title('Consistency Models: All points on same trajectory (same colour)\n'
             'map to the SAME endpoint (★)', fontsize=12)
ax.set_xlim(-4, 4); ax.set_ylim(-4, 4); ax.set_aspect('equal')
plt.tight_layout(); plt.show()
print('A consistency model learns f(x_t, t) = x_0 for ALL points on a trajectory.')
print('Single-step generation: evaluate f(x_T, T) once. No iteration!')

---

## 6. Grand Comparison

### 6.1 Unified Comparison Table

| Feature | VAE | GAN | DDPM | DDIM | Score/SDE | Flow Matching | Rectified Flow | Consistency |
|---|---|---|---|---|---|---|---|---|
| **Training** | Maximise ELBO | Minimax game | Denoise ($\epsilon$ prediction) | Same model | Score matching | Velocity MSE | Velocity MSE (reflow) | Self-consistency |
| **Sampling** | 1 step (decode) | 1 step (generate) | $T$ steps | $S \ll T$ steps | SDE/ODE solve | ODE solve | ODE (fewer steps) | 1-2 steps |
| **Quality** | Blurry | Sharp | Excellent | Good-Excellent | Excellent | Excellent | Excellent | Good-Excellent |
| **Stability** | Stable | Unstable | Stable | N/A | Stable | Stable | Stable | Stable |
| **Mode coverage** | Good | Collapse risk | Good | Good | Good | Good | Good | Good |
| **Density est.** | Yes (ELBO) | No | Yes (ELBO) | No | Yes (via ODE) | Yes (via ODE) | Yes (via ODE) | No |
| **Year** | 2013 | 2014 | 2020 | 2020 | 2020-21 | 2022-23 | 2022-23 | 2023 |

### 6.2 The Efficiency Trajectory

```
VAE / GAN:       1 step    (but limited quality or stability)
DDPM:            ~1000 steps (breakthrough quality)
DDIM:            ~50 steps   (same model, faster)
Flow Matching:   ~10-50 steps (straighter paths)
Rectified Flow:  ~5-10 steps  (even straighter)
Consistency:     1-2 steps    (best of both worlds)
```

The story of generative models is one of **progressively recovering single-step speed** while maintaining the quality breakthrough that diffusion brought:

```
Phase 1: VAE + GAN (1 step, limited)
   ↓ quality breakthrough
Phase 2: DDPM (1000 steps, excellent)
   ↓ fewer steps
Phase 2: DDIM (50 steps)
   ↓ mathematical unification
Phase 3: Score/SDE/ODE (continuous framework)
   ↓ straighter paths
Phase 4: Flow Matching (10-50 steps)
   ↓ single step
Phase 5: Consistency (1 step, excellent)
```

---
# Part B · In-Class Coding Practice

You have just seen a complete DDPM in Cells 14–19. Your task is to **rebuild the three lines of code that make a diffusion model work** — forward noising, the ε-prediction loss, and one DDIM update step. This exercise has nothing hidden: every formula you need is on the slides above.

**Why these three?** They are the entire technical content shared by DDPM, DDIM, Score-SDE, Flow Matching (with a different parameterisation), and Consistency models. If you can write them from memory, you can read any modern diffusion paper.

**Rules of the game.**
- Do **not** look at Cells 16 or 18/21 while you write — those are the answer key.
- Use only `torch`, `math`, and the helpers already provided in the cell.
- When you are done, run the visualisation cell. Your samples should land near the 8 modes after ~800 training steps.

The **reference solution cell is directly below** — do not expand it until the tutor walks through it.

In [ ]:
# ===== Section P: In-class practice (DO NOT peek at the solution cell) =====
#
# Tasks:
#   T1. Implement the forward noising q_sample(x0, t).
#   T2. Implement the ε-prediction training loop.
#   T3. Implement one deterministic DDIM update step (eta = 0).
#
# All variables you need are constructed for you below.
# Replace each `raise NotImplementedError` with your own code.

torch.manual_seed(0)
practice_T = 200
practice_betas  = linear_beta_schedule(practice_T)         # from Cell 14
practice_alphas = 1.0 - practice_betas
practice_abars  = torch.cumprod(practice_alphas, dim=0)

class TinyNoiseNet(nn.Module):
    """A small ε-predictor. Re-uses SinusoidalTimeEmb from Cell 16."""
    def __init__(self, hidden=128, time_dim=32):
        super().__init__()
        self.temb = SinusoidalTimeEmb(time_dim)
        self.net = nn.Sequential(
            nn.Linear(2 + time_dim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden),       nn.SiLU(),
            nn.Linear(hidden, 2))
    def forward(self, x, t):
        return self.net(torch.cat([x, self.temb(t)], dim=-1))

practice_model = TinyNoiseNet()
practice_opt   = torch.optim.Adam(practice_model.parameters(), lr=2e-3)

# ---------- T1. Forward noising ----------
def q_sample(x0, t, abars):
    """
    Return (x_t, eps) such that
        x_t = sqrt(abars[t]) * x0 + sqrt(1 - abars[t]) * eps,   eps ~ N(0, I).
    Shapes:
        x0:    (B, 2)
        t:     (B,) integer tensor
        abars: (T,)
    Hints:
      - Index abars with t to get a (B,) tensor, then unsqueeze(-1) for broadcasting.
      - Use torch.randn_like(x0) for eps.
    """
    # TODO ↓
    raise NotImplementedError

# ---------- T2. Training loop ----------
for step in range(800):
    idx = torch.randperm(len(data_2d))[:256]
    x0  = data_2d[idx]
    t   = torch.randint(0, practice_T, (256,))

    # TODO ↓
    # (a) x_t, eps = q_sample(...)
    # (b) eps_pred = practice_model(x_t, t.float())
    # (c) loss = MSE(eps_pred, eps)
    # (d) zero_grad / backward / step
    raise NotImplementedError

    if (step + 1) % 200 == 0:
        print(f'  practice step {step+1}, loss={loss.item():.4f}')

# ---------- T3. One DDIM update step ----------
@torch.no_grad()
def ddim_step(x_t, t_int, t_prev_int, model, abars):
    """
    Deterministic DDIM update (eta = 0):
        eps_hat = model(x_t, t)
        x0_hat  = (x_t - sqrt(1 - a_t) * eps_hat) / sqrt(a_t)
        x_prev  = sqrt(a_prev) * x0_hat + sqrt(1 - a_prev) * eps_hat
    where a_t = abars[t_int]; a_prev = abars[t_prev_int] if t_prev_int >= 0 else 1.0.
    `t_int` is an int (same for the whole batch), pass it to the model as a float tensor.
    """
    # TODO ↓
    raise NotImplementedError

# ---------- Quick sampler that uses your ddim_step ----------
@torch.no_grad()
def practice_sample(model, n=1000, S=20):
    ts = torch.linspace(practice_T - 1, 0, S + 1).long().tolist()
    x = torch.randn(n, 2)
    for i in range(S):
        x = ddim_step(x, ts[i], ts[i + 1] if i + 1 < len(ts) else -1,
                      model, practice_abars)
    return x

samples = practice_sample(practice_model, n=1500, S=20)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(data_2d[:1500, 0], data_2d[:1500, 1], s=3, alpha=0.25, label='real')
ax.scatter(samples[:, 0], samples[:, 1], s=3, alpha=0.5, c='red',
           label='your DDIM (20 steps)')
ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5); ax.set_aspect('equal')
ax.legend(); ax.set_title('Section P — your samples')
plt.show()


### Section P — Reference solution

> The tutor will walk through this after the practice. Try not to look until you have something running.


In [ ]:
# ===== Section P: reference solution =====
torch.manual_seed(0)

def q_sample(x0, t, abars):
    ab = abars[t].unsqueeze(-1)            # (B, 1)
    eps = torch.randn_like(x0)
    x_t = torch.sqrt(ab) * x0 + torch.sqrt(1.0 - ab) * eps
    return x_t, eps

practice_model = TinyNoiseNet()
practice_opt   = torch.optim.Adam(practice_model.parameters(), lr=2e-3)

for step in range(800):
    idx = torch.randperm(len(data_2d))[:256]
    x0  = data_2d[idx]
    t   = torch.randint(0, practice_T, (256,))
    x_t, eps = q_sample(x0, t, practice_abars)
    eps_pred = practice_model(x_t, t.float())
    loss = F.mse_loss(eps_pred, eps)
    practice_opt.zero_grad(); loss.backward(); practice_opt.step()
    if (step + 1) % 200 == 0:
        print(f'  practice step {step+1}, loss={loss.item():.4f}')

@torch.no_grad()
def ddim_step(x_t, t_int, t_prev_int, model, abars):
    a_t    = abars[t_int]
    a_prev = abars[t_prev_int] if t_prev_int >= 0 else torch.tensor(1.0)
    t_batch = torch.full((x_t.shape[0],), float(t_int))
    eps_hat = model(x_t, t_batch)
    x0_hat  = (x_t - torch.sqrt(1.0 - a_t) * eps_hat) / torch.sqrt(a_t)
    return torch.sqrt(a_prev) * x0_hat + torch.sqrt(1.0 - a_prev) * eps_hat

samples = practice_sample(practice_model, n=1500, S=20)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(data_2d[:1500, 0], data_2d[:1500, 1], s=3, alpha=0.25, label='real')
ax.scatter(samples[:, 0], samples[:, 1], s=3, alpha=0.5, c='green',
           label='reference solution (20 steps)')
ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5); ax.set_aspect('equal')
ax.legend(); ax.set_title('Section P — reference solution')
plt.show()


---
# Part C · Exam-Style Questions

Three short-answer questions at mid-to-upper exam difficulty. Each combines this week's content with foundations from earlier weeks (probability, KL divergence, MSE training, autograd). Marks in brackets are indicative. Answer sketches are provided in collapsed cells directly below each question — expand them only after attempting.

### Q1 · ELBO and the reparameterisation trick  *(7 marks)*

A VAE has encoder $q_\phi(z\mid x)=\mathcal{N}\!\left(\mu_\phi(x),\,\mathrm{diag}\,\sigma_\phi^2(x)\right)$, decoder $p_\theta(x\mid z)$, and prior $p(z)=\mathcal{N}(0,I)$.

**(a)** *(3 marks)* Starting from $\log p_\theta(x)$, derive the ELBO
$$\log p_\theta(x)\;\ge\;\mathbb{E}_{q_\phi(z\mid x)}\big[\log p_\theta(x\mid z)\big]\;-\;\mathrm{KL}\!\left(q_\phi(z\mid x)\,\Vert\,p(z)\right).$$
State the inequality direction and identify which term is the *reconstruction* loss and which is the *regulariser*.

**(b)** *(2 marks)* Suppose, instead of writing $z=\mu_\phi(x)+\sigma_\phi(x)\odot\varepsilon$ with $\varepsilon\sim\mathcal{N}(0,I)$, we wrote `z = torch.normal(mu, sigma)` directly inside the forward pass. Explain precisely **which** part of backpropagation breaks, and why the resulting gradient with respect to $\phi$ would be unbiased only after a high-variance estimator (e.g. REINFORCE) is added.

**(c)** *(2 marks)* For $q_\phi=\mathcal{N}(\mu,\mathrm{diag}\,\sigma^2)$ and prior $\mathcal{N}(0,I)$, write the closed-form KL divergence used in the VAE loss (the formula that appears in `vae_loss` in Cell 5).

<details><summary><b>▸ Answer sketch — Q1</b></summary>

**(a)** Introduce $q_\phi(z\mid x)$ inside the log-likelihood integral:
$$\log p_\theta(x) = \log \int p_\theta(x,z)\,dz = \log \int q_\phi(z\mid x)\,\frac{p_\theta(x,z)}{q_\phi(z\mid x)}\,dz.$$
Apply Jensen's inequality (the log is concave, so the log of an expectation is ≥ the expectation of the log):
$$\log p_\theta(x) \ge \mathbb{E}_{q_\phi}\!\left[\log \frac{p_\theta(x, z)}{q_\phi(z\mid x)}\right] = \mathbb{E}_{q_\phi}[\log p_\theta(x\mid z)] + \mathbb{E}_{q_\phi}[\log p(z) - \log q_\phi(z\mid x)].$$
The second term is exactly $-\mathrm{KL}(q_\phi(z\mid x) \,\|\, p(z))$, giving
$$\log p_\theta(x) \ge \underbrace{\mathbb{E}_{q_\phi}[\log p_\theta(x\mid z)]}_{\text{reconstruction}} - \underbrace{\mathrm{KL}(q_\phi(z\mid x) \,\|\, p(z))}_{\text{regulariser}}.$$
The gap between the two sides is $\mathrm{KL}(q_\phi\,\|\,p_\theta(z\mid x)) \ge 0$, so the ELBO is always a **lower** bound on the log-likelihood — maximising it maximises a guaranteed lower bound on $\log p_\theta(x)$.

**(b)** `torch.normal(mu, sigma)` draws a sample whose value depends on $\mu, \sigma$ but **without** a differentiable computation graph linking them — the sampling operation has no gradient path from the output back through $\mu$ and $\sigma$. Concretely, the operation `z = sample(N(mu, sigma))` is equivalent to "look up a random number, ignore how $\mu$ and $\sigma$ affect the randomness" — the chain rule has nothing to differentiate. The reparameterisation trick moves the randomness *outside* the computation graph: $\varepsilon$ is sampled from a *fixed* $\mathcal{N}(0, I)$, and $z = \mu + \sigma \odot \varepsilon$ is a smooth, differentiable function of the encoder outputs. Without this trick, the only unbiased gradient estimator is the **score-function** (REINFORCE) estimator $\nabla_\phi \mathbb{E}_{q_\phi}[f(z)] = \mathbb{E}_{q_\phi}[f(z)\,\nabla_\phi \log q_\phi(z\mid x)]$, which is unbiased but has variance that scales poorly with dimension — training a VAE with REINFORCE is prohibitively noisy in practice.

**(c)** For diagonal-covariance Gaussians,
$$\mathrm{KL}(\mathcal{N}(\mu, \mathrm{diag}\,\sigma^2) \,\|\, \mathcal{N}(0, I)) = \frac{1}{2}\sum_{i=1}^{d}\!\left(\mu_i^2 + \sigma_i^2 - \log \sigma_i^2 - 1\right).$$
In the code this is typically written with $\log \sigma^2$ (`logvar`) stored directly for numerical stability:
```python
kl = 0.5 * (mu.pow(2) + logvar.exp() - logvar - 1).sum(dim=-1).mean()
```
</details>

### Q2 · ε-prediction *is* score matching  *(7 marks)*

DDPM trains a network $\varepsilon_\theta(x_t,t)$ with the loss
$$L(\theta)=\mathbb{E}_{x_0,\varepsilon,t}\!\left[\,\Vert\varepsilon_\theta(x_t,t)-\varepsilon\Vert^2\,\right],
\qquad x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\varepsilon.$$

**(a)** *(3 marks)* Using $q(x_t\mid x_0)=\mathcal{N}\!\left(\sqrt{\bar\alpha_t}\,x_0,\,(1-\bar\alpha_t)I\right)$, show that
$$\nabla_{x_t}\log q(x_t\mid x_0)\;=\;-\frac{x_t-\sqrt{\bar\alpha_t}\,x_0}{1-\bar\alpha_t}\;=\;-\frac{\varepsilon}{\sqrt{1-\bar\alpha_t}}.$$

**(b)** *(2 marks)* Hence write the formula that converts a trained noise predictor $\varepsilon_\theta(x_t,t)$ into an estimate of the **marginal** score $s_\theta(x_t,t)\approx\nabla_{x_t}\log p_t(x_t)$.

**(c)** *(2 marks)* DDPM could equivalently be parameterised to predict $x_0$ directly or to predict the posterior mean $\mu_\theta$. Why is **ε-prediction** preferred in practice? Answer in terms of how the *target* $\varepsilon$ behaves across timesteps compared to the targets $x_0$ or $\mu$ (think about the loss landscape near $t\to 0$ and $t\to T$).

<details><summary><b>▸ Answer sketch — Q2</b></summary>

**(a)** For a Gaussian $q(x_t\mid x_0) = \mathcal{N}(\mu, \sigma^2 I)$ with $\mu = \sqrt{\bar\alpha_t}\,x_0$ and $\sigma^2 = 1 - \bar\alpha_t$, the log-density is
$$\log q(x_t \mid x_0) = -\frac{\Vert x_t - \mu\Vert^2}{2\sigma^2} + \text{const}.$$
Taking the gradient with respect to $x_t$:
$$\nabla_{x_t} \log q(x_t \mid x_0) = -\frac{x_t - \sqrt{\bar\alpha_t}\, x_0}{1 - \bar\alpha_t}.$$
Using the forward equation $x_t - \sqrt{\bar\alpha_t}\, x_0 = \sqrt{1 - \bar\alpha_t}\, \varepsilon$, this simplifies to
$$\nabla_{x_t} \log q(x_t \mid x_0) = -\frac{\sqrt{1 - \bar\alpha_t}\, \varepsilon}{1 - \bar\alpha_t} = -\frac{\varepsilon}{\sqrt{1 - \bar\alpha_t}}. \qquad \blacksquare$$
The denoising score-matching objective trains $s_\theta$ to match $\nabla_{x_t}\log q(x_t\mid x_0)$ in expectation over $(x_0, \varepsilon)$; since the *marginal* score equals the *conditional* score under this expectation, this also matches $\nabla_{x_t}\log p_t(x_t)$.

**(b)** Rearranging part (a):
$$s_\theta(x_t, t) = -\frac{\varepsilon_\theta(x_t, t)}{\sqrt{1 - \bar\alpha_t}}.$$
This is the formula used when plugging a DDPM-trained network into a score-based SDE sampler, a PF-ODE solver, or Langevin dynamics — the weights trained with ε-prediction are *directly re-usable*.

**(c)** The key observation is that the target $\varepsilon \sim \mathcal{N}(0, I)$ is **timestep-independent** — it always has unit variance and zero mean, regardless of whether $t \to 0$ (nearly clean data) or $t \to T$ (nearly pure noise). This gives the loss a **uniform scale** across all timesteps and a well-behaved, homogeneous loss landscape. In contrast, predicting $x_0$ directly means the target scale is the data scale — and the coefficient relating the network output to the residual becomes $\sqrt{\bar\alpha_t}$, which shrinks to zero at $t \to T$: the signal is buried in noise and the network cannot learn anything at high $t$. Predicting $\mu_\theta$ (posterior mean) has a similar problem because $\mu$ scales with $\sqrt{\alpha_t}$ across timesteps. Empirically, ε-prediction gives a roughly constant-scale loss that can be trained with a single learning rate across all $t$ — this is the main pragmatic reason Ho et al. (2020) adopted it, and why almost every diffusion model since has followed suit.
</details>

### Q3 · From 1000 steps to 1 step  *(8 marks)*

You have a fully trained DDPM with $T=1000$ noise steps. At inference time you can afford only **4** network evaluations.

**(a)** *(2 marks)* Explain why running the *DDPM* sampler for 4 of its 1000 steps would produce garbage, but running **DDIM** with $S=4$ subsampled timesteps is at least plausible. Which property of the DDIM update rule allows the same trained model to be re-used at a different step count?

**(b)** *(3 marks)* State the **probability flow ODE (PF-ODE)** that has the same marginals $\{p_t\}$ as the DDPM forward SDE, in terms of the score $\nabla_{x_t}\log p_t(x_t)$. Using your answer to **Q2(b)**, rewrite the PF-ODE so the velocity is expressed in terms of the trained network $\varepsilon_\theta$ — i.e. the form actually used in code.

**(c)** *(3 marks)* Define the **self-consistency** property that a consistency model $f_\theta(x_t,t)$ must satisfy along a single PF-ODE trajectory, and explain in 2–3 sentences how enforcing it (via consistency distillation from your DDPM) yields a model that samples in **one** step. State the **boundary condition** $f_\theta$ must satisfy at $t=0$ and describe how it is usually enforced *architecturally* rather than as a soft loss term.

<details><summary><b>▸ Answer sketch — Q3</b></summary>

**(a)** The DDPM sampler implements a **Markovian** reverse chain $p_\theta(x_{t-1} \mid x_t)$ calibrated to $T = 1000$: each step subtracts a carefully-sized increment of noise based on the *local* transition $q(x_{t-1} \mid x_t, x_0)$. Skipping steps wholesale violates the local Markov assumption — going from $t = 1000$ to $t = 750$ in one DDPM step ignores all of the intermediate conditional structure, and you end up sampling a badly-misspecified approximate posterior. DDIM instead defines a **non-Markovian deterministic update** that expresses $x_{t_{i-1}}$ directly in terms of $x_{t_i}$ and $\varepsilon_\theta(x_{t_i}, t_i)$:
$$x_{t_{i-1}} = \sqrt{\bar\alpha_{t_{i-1}}}\, \hat x_0 + \sqrt{1 - \bar\alpha_{t_{i-1}}}\, \varepsilon_\theta(x_{t_i}, t_i), \quad \hat x_0 = \frac{x_{t_i} - \sqrt{1 - \bar\alpha_{t_i}}\, \varepsilon_\theta(x_{t_i}, t_i)}{\sqrt{\bar\alpha_{t_i}}}.$$
This update is valid for **any** subsequence $t_0 < t_1 < \dots < t_S$ of the original 1000 timesteps because it only depends on the learned noise predictor $\varepsilon_\theta$ and the cumulative schedule $\bar\alpha_t$ at the chosen indices — no Markovian spacing assumption is required. Same trained model, different number of evaluations.

**(b)** For the variance-preserving DDPM SDE $dx = -\tfrac{1}{2}\beta(t)\,x\,dt + \sqrt{\beta(t)}\,dW$, Song et al. (2021) showed the corresponding PF-ODE (which preserves the same marginals $p_t$) is
$$\frac{dx}{dt} = -\frac{1}{2}\beta(t)\,x - \frac{1}{2}\beta(t)\,\nabla_{x_t}\log p_t(x_t).$$
Substituting the score expression from Q2(b):
$$\frac{dx}{dt} = -\frac{1}{2}\beta(t)\,x + \frac{\beta(t)}{2\sqrt{1 - \bar\alpha_t}}\,\varepsilon_\theta(x, t).$$
This is the form that actually gets integrated in code — an ODE whose right-hand side is one network evaluation. Picking any ODE solver (Euler, Heun, DPM-Solver, etc.) and any number of steps turns a diffusion model into a deterministic generator.

**(c)** A consistency model $f_\theta(x_t, t)$ satisfies the **self-consistency property**: for **any** two points $(x_{t_1}, t_1)$ and $(x_{t_2}, t_2)$ lying on the *same* PF-ODE trajectory, $f_\theta(x_{t_1}, t_1) = f_\theta(x_{t_2}, t_2) = x_0$. In other words, every point on the trajectory maps to the same data endpoint under $f_\theta$. Consistency distillation trains $f_\theta$ to satisfy this by (i) generating a pair of adjacent points on the trajectory using a teacher DDPM + an ODE solver and (ii) penalising $\lVert f_\theta(x_{t_1}, t_1) - f_{\theta^-}(x_{t_2}, t_2)\rVert$ where $\theta^-$ is an EMA target. Once trained, sampling is one step: draw $x_T \sim \mathcal{N}(0, I)$ and return $f_\theta(x_T, T)$ — the self-consistency property guarantees the single-step answer equals what a full ODE solve would give.

**Boundary condition:** $f_\theta(x_0, 0) = x_0$ (the identity on clean data). This is enforced *architecturally* via a skip connection with timestep-dependent coefficients:
$$f_\theta(x, t) = c_{\text{skip}}(t)\, x + c_{\text{out}}(t)\, F_\theta(x, t),$$
with $c_{\text{skip}}(0) = 1$ and $c_{\text{out}}(0) = 0$. Any network output $F_\theta$ at $t = 0$ is multiplied away, so the boundary condition holds *by construction* — no soft loss term can undo it, and training is stable from step 1.
</details>